In [42]:
import pandas as pd
import numpy as np

In [43]:
staging = pd.read_csv('staging_dataset_cmu 1.csv')

In [44]:
staging.head()

,ICD10_CODE,member_number,MOST_RECENT_PATH_STAGE_DT,MOST_RECENT_CLINICAL_STAGE_DT,PATHOLOGIC_STAGE_GROUP,CLINICAL_STAGE_GROUP
0,C50.411,B908,64.0,NaN,Stage IA,NaN
1,C50.919,B760,NaN,888.0,NaN,Stage IA
2,C34.11,C597,NaN,0.0,NaN,Stage IVB
3,C50.519,B621,77.0,NaN,Stage IA,NaN
4,C50.412,A392,33.0,NaN,Stage IA,NaN


In [45]:
staging.info()

<class 'pandas.DataFrame'>
RangeIndex: 3767 entries, 0 to 3766
Data columns (total 6 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   ICD10_CODE                     3767 non-null   str    
 1   member_number                  3767 non-null   str    
 2   MOST_RECENT_PATH_STAGE_DT      1913 non-null   float64
 3   MOST_RECENT_CLINICAL_STAGE_DT  2313 non-null   float64
 4   PATHOLOGIC_STAGE_GROUP         1853 non-null   str    
 5   CLINICAL_STAGE_GROUP           2223 non-null   str    
dtypes: float64(2), str(4)
memory usage: 251.5 KB


In [46]:
# how many member numbers have multiple records in staging
(staging['member_number'].value_counts() > 1).sum()

107

In [47]:
# remove members with multiple records in staging
counts = staging['member_number'].value_counts()

single_members = counts[counts == 1].index

staging = staging[staging['member_number'].isin(single_members)]

In [48]:
staging.shape

(3550, 6)

In [49]:
# Create most_recent_stage_date
# 1. If only one of the two columns is non-null → take that value
# 2. If both are non-null → take the larger value
# 3. If both are null → result will remain NaN

staging['most_recent_stage_date'] = staging[
    ['MOST_RECENT_PATH_STAGE_DT', 'MOST_RECENT_CLINICAL_STAGE_DT']
].max(axis=1)

staging

,ICD10_CODE,member_number,MOST_RECENT_PATH_STAGE_DT,MOST_RECENT_CLINICAL_STAGE_DT,PATHOLOGIC_STAGE_GROUP,CLINICAL_STAGE_GROUP,most_recent_stage_date
0,C50.411,B908,64.0,NaN,Stage IA,NaN,64.0
1,C50.919,B760,NaN,888.0,NaN,Stage IA,888.0
2,C34.11,C597,NaN,0.0,NaN,Stage IVB,0.0
3,C50.519,B621,77.0,NaN,Stage IA,NaN,77.0
4,C50.412,A392,33.0,NaN,Stage IA,NaN,33.0
...,...,...,...,...,...,...,...
3760,C50.211,C418,NaN,-705.0,NaN,Stage IB,-705.0
3761,C34.32,C277,NaN,0.0,NaN,Stage IA2,0.0
3762,C20,D338,NaN,-2168.0,NaN,Stage IIA,-2168.0
3763,C50.812,B004,NaN,238.0,NaN,Stage IIA,238.0


In [50]:
# Create final stage_group column
# 1. Prefer PATHOLOGIC_STAGE_GROUP if present
# 2. Otherwise use CLINICAL_STAGE_GROUP
# 3. If both are null → result remains null

staging['stage_group'] = staging['PATHOLOGIC_STAGE_GROUP'].combine_first(
    staging['CLINICAL_STAGE_GROUP']
)

In [51]:
# show value counts for final stage_group column
staging['stage_group'].value_counts(dropna=False)

stage_group
Stage IA                1180
Stage IIA                387
Stage IIIB               320
Stage IB                 310
Stage IIB                236
Stage IIIA               181
Stage IV                 178
No Stage Recommended     131
Stage IA2                113
Stage IVA                111
Stage IIIC                95
Stage I                   94
Stage IVB                 79
Stage IA3                 46
Stage IA1                 27
Stage IVC                 25
Stage 0                   15
Stage Unknown             11
Stage IIC                  7
Stage II                   3
Occult carcinoma           1
Name: count, dtype: int64

In [52]:
# Keep only stages that start with I, II, III, or IV
# Remove:
# - No Stage Recommended
# - Stage 0
# - Stage Unknown
# - Occult carcinoma

staging = staging[
    staging['stage_group'].str.startswith(
        ('Stage I', 'Stage II', 'Stage III', 'Stage IV'),
        na=False  # Ensures NaN rows are excluded
    )
]

In [53]:
# Create stage_source column
staging['stage_source'] = np.where(
    staging['PATHOLOGIC_STAGE_GROUP'].notna(),
    'pathologic',
    np.where(
        staging['CLINICAL_STAGE_GROUP'].notna(),
        'clinical',
        np.nan
    )
)

In [54]:
# Create simplified ICD10 group (first 3 characters)
staging['ICD10_simple'] = staging['ICD10_CODE'].str[:3]

cancer_map = {
    "C50": "breast",
    "C34": "lung",
    "C18": "colorectal",
    "C19": "colorectal",
    "C20": "colorectal",
    "C21": "colorectal"
}

staging['cancer_type'] = staging['ICD10_simple'].map(cancer_map)

In [55]:
# Create simplified stage (I, II, III, IV)

staging['final_stage_simple'] = (
    staging['stage_group']
        .str.extract(r'Stage (IV|III|II|I)')[0]
)

In [56]:
staging['final_stage_simple'].value_counts(dropna=False)

final_stage_simple
I      1770
II      633
III     596
IV      393
Name: count, dtype: int64

In [57]:
staging.columns

Index(['ICD10_CODE', 'member_number', 'MOST_RECENT_PATH_STAGE_DT',
       'MOST_RECENT_CLINICAL_STAGE_DT', 'PATHOLOGIC_STAGE_GROUP',
       'CLINICAL_STAGE_GROUP', 'most_recent_stage_date', 'stage_group',
       'stage_source', 'ICD10_simple', 'cancer_type', 'final_stage_simple'],
      dtype='str')

In [58]:
# drop intermediate columns that are no longer needed
staging.drop(
    columns=[
        'MOST_RECENT_PATH_STAGE_DT',
        'MOST_RECENT_CLINICAL_STAGE_DT',
        'PATHOLOGIC_STAGE_GROUP',
        'CLINICAL_STAGE_GROUP'
    ],
    inplace=True
)

In [59]:
# standardize to match claims
staging['ICD10_CODE'] = (
    staging['ICD10_CODE']
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace('.', '', regex=False)
)

In [60]:
# re-order columns for better readability
staging = staging[
    [
        'member_number',
        'ICD10_CODE',
        'ICD10_simple',
        'cancer_type',
        'stage_group',
        'final_stage_simple',
        'stage_source',
        'most_recent_stage_date'
    ]
]

staging.head()

,member_number,ICD10_CODE,ICD10_simple,cancer_type,stage_group,final_stage_simple,stage_source,most_recent_stage_date
0,B908,C50411,C50,breast,Stage IA,I,pathologic,64.0
1,B760,C50919,C50,breast,Stage IA,I,clinical,888.0
2,C597,C3411,C34,lung,Stage IVB,IV,clinical,0.0
3,B621,C50519,C50,breast,Stage IA,I,pathologic,77.0
4,A392,C50412,C50,breast,Stage IA,I,pathologic,33.0


In [61]:
staging.info()

<class 'pandas.DataFrame'>
Index: 3392 entries, 0 to 3764
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   member_number           3392 non-null   str    
 1   ICD10_CODE              3392 non-null   str    
 2   ICD10_simple            3392 non-null   str    
 3   cancer_type             3392 non-null   str    
 4   stage_group             3392 non-null   str    
 5   final_stage_simple      3392 non-null   str    
 6   stage_source            3392 non-null   str    
 7   most_recent_stage_date  3392 non-null   float64
dtypes: float64(1), str(7)
memory usage: 366.5 KB


In [62]:
# export cleaned staging dataset
staging.to_csv('cleaned_staging.csv', index=False)